<a href="https://colab.research.google.com/github/cras-lab/LangChain/blob/main/RAG%EC%8B%A4%EC%A0%84%EC%82%AC%EB%A1%80_ToyE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

자신의 모델을 선택한다.

In [1]:
LLM_PROVIDER = "openai" # claude 또는 gemini
TEMPERATURE = 0.2

필요한 모듈을 설치한다. LangChain을 이용한다.

In [ ]:
pip install -q -U langchain langchain-openai langchain-anthropic langchain-google-genai langchain-community wikipedia



> LangChain에서 각종 모듈을 임포트 한다.



In [ ]:
from langchain_openai import ChatOpenAI
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

키를 입력받는다.

In [4]:
import getpass
API_KEY = getpass.getpass("Enter your API_KEY: ")

Enter your API_KEY: ··········


비용이 가장 저렴한 GPT5 nano 버전을 이용한다.<BR>
+ RAG 성능비교를 극대화하기 위해 가급적 성능 낮은버전 사용한다.

In [7]:
if LLM_PROVIDER == "openai":
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(
        model="gpt-5-nano",
        temperature=TEMPERATURE,
        api_key=API_KEY,
      )

elif LLM_PROVIDER == "claude":
    from langchain_anthropic import ChatAnthropic

    llm = ChatAnthropic(
        model="claude-sonnet-4-6",
        temperature=TEMPERATURE,
        api_key=API_KEY,
      )


elif LLM_PROVIDER == "gemini":
    from langchain_google_genai import ChatGoogleGenerativeAI

    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=TEMPERATURE,
        api_key=API_KEY,
      )


질문을 구성한다.

In [8]:
question = "이병욱 교수에 대해 알려줘"

RAG 없이 바로 질문해 본다.

In [9]:
no_rag_prompt = ChatPromptTemplate.from_messages([
    ("system", # System 지시문은 통상 여기 작성한다. role 부여
     "너는 정확성을 우선하는 assistant다. 모르면 모른다고 말하라."),
    ("human", "{question}")
])

질문하기 위한 Chain을 구성한다.

In [10]:
no_rag_chain = no_rag_prompt | llm | StrOutputParser()

체인을 호출해서 대답을 얻어 온다.

In [11]:
no_rag_answer = no_rag_chain.invoke({"question": question})

출력해 보자.

In [ ]:
print( no_rag_answer )

이제 RAG를 위해 위키피디아에서 질의해서 결과를가져오자

In [ ]:
wiki = WikipediaAPIWrapper(
    lang="ko",        # 한국어 위키 우선
    top_k_results=3,
    doc_content_chars_max=4000
)

wiki_context = wiki.run("이병욱 교수")

위키피디아에서 수집한 정보를 살펴보자.

In [ ]:
print(wiki_context[:3000])  # 너무 길면 일부만 출력

RAG를 적용할 새로운 프롬프트의 템플릿과 지침을 설정한다.

In [15]:
rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system", # System 지시문은 통상 여기 작성한다. role 부여
        "너는 검색 기반 질의응답 assistant다. "
        "반드시 제공된 CONTEXT만 근거로 답해. "
        "CONTEXT에 없는 내용은 추정하지 말고, 확인 불가라고 말해. "
        "답변 마지막에 '근거 요약'을 2~4줄로 붙여."
    ),
    (
        "human", # 사용자 메시지는 통상 여기 작성한다.
        "질문:\n{question}\n\n"
        "CONTEXT:\n{context}\n\n"  # {context}는 invikeEo wiki_context로 대체
        "위 정보를 바탕으로 답하라."
    )
])

RAG로 얻은 정보를 입력해, 답을 받기위한 Chain을 구성한다.

In [16]:
rag_chain = rag_prompt | llm | StrOutputParser()

체인을 실행시킨다.

In [17]:
rag_answer = rag_chain.invoke({
    "question": question,
    "context": wiki_context
})

최종출력과 함께 이전 출력도 다시 모아 비교해 보자.<BR>
[1] RAG없는 버전 [2] Wiki 추출버전 [3] Wiki를 RAG로 반영

In [ ]:
# 첫번째 결과 다시
print("=" * 80)
print("[1] RAG를 사용하지 않은 경우")
print("=" * 80)
print(no_rag_answer)

# Wikipedia 정보
print("\n" + "=" * 80)
print("[2] Wikipedia 검색 결과")
print("=" * 80)
print(wiki_context[:3000])

#최종 출력
print("\n" + "=" * 80)
print("[3] RAG를 적용 후 결과")
print("=" * 80)
print(rag_answer)